[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vpasumarthi/aimd-tutorial/blob/main/notebooks/01_bulk_water.ipynb)

# Bulk Liquid Water: AIMD Setup and Analysis

This tutorial walks through setting up and analyzing an ab initio molecular dynamics (AIMD) simulation of bulk liquid water. We use a periodic cell containing 32 H$_2$O molecules under NVT conditions.

**Learning objectives:**
- Configure VASP input files for NVT AIMD of liquid water
- Verify thermal equilibration from the trajectory
- Compute radial distribution functions (O-O, O-H, H-H)
- Calculate mean squared displacement and extract the diffusion coefficient

In [ ]:
# --- Run this cell first in Google Colab ---
import sys
if 'google.colab' in sys.modules:
    %pip install -q ase pymatgen nglview scipy
    from google.colab import output
    output.enable_custom_widget_manager()

## Background

Liquid water is one of the most-studied systems in AIMD. DFT-based simulations can capture bond breaking/forming, polarization, and charge transfer that classical force fields miss. However, standard GGA functionals (PBE) over-structure liquid water at 300 K, so AIMD of water is typically run at 330-400 K to better reproduce experimental structure and dynamics at ambient conditions.

Our system: 32 H$_2$O molecules in a cubic cell with side length ~9.86 Angstrom, chosen to match the experimental density of 1.0 g/cm$^3$. We run NVT dynamics with a Nose-Hoover thermostat at 400 K.

## VASP Input Files

### INCAR

```
SYSTEM = Bulk liquid water - 32 H2O NVT

# Electronic structure
PREC    = Normal
ENCUT   = 400        # Plane-wave cutoff (eV)
ALGO    = VeryFast   # RMM-DIIS for fast SCF convergence
EDIFF   = 1E-5       # SCF convergence criterion (eV)
ISMEAR  = 0          # Gaussian smearing (insulator)
SIGMA   = 0.1        # Smearing width (eV)
LREAL   = Auto       # Real-space projection
NELMIN  = 4          # Minimum SCF steps (stabilizes MD)

# Molecular dynamics
IBRION  = 0          # MD mode
MDALGO  = 2          # Nose-Hoover thermostat (NVT)
SMASS   = 1.0        # Nose mass parameter (controls coupling strength)
POTIM   = 1.0        # Timestep in fs
NSW     = 5000       # Number of MD steps (5 ps total)
TEBEG   = 400        # Starting temperature (K)
TEEND   = 400        # Final temperature (K)
ISIF    = 2          # Fixed cell shape and volume (NVT)

# Output control
NWRITE  = 0          # Minimal OUTCAR output
NBLOCK  = 1          # Write trajectory every step
LWAVE   = .FALSE.    # Do not write WAVECAR
LCHARG  = .FALSE.    # Do not write CHGCAR
```

**Key choices:**
- `MDALGO = 2` selects the Nose-Hoover thermostat. `SMASS` controls the coupling -- values of 0.5-3.0 are typical. Larger values give weaker coupling and longer temperature relaxation times.
- `POTIM = 1.0` fs is standard for water AIMD. Lighter elements (H) set the upper limit on the timestep.
- `TEBEG = 400` K compensates for the known over-structuring of PBE water at 300 K.
- `NELMIN = 4` forces at least 4 SCF iterations per step, preventing premature convergence that can cause energy drift.

### POSCAR

The initial structure is 32 H$_2$O molecules in a cubic box of side 9.86 Angstrom (density = 1.0 g/cm$^3$). You can generate this with [Packmol](http://leandro.iqm.unicamp.br/m3g/packmol/home.shtml) or from a classical MD pre-equilibration.

```
Bulk liquid water - 32 H2O
1.0
  9.8600  0.0000  0.0000
  0.0000  9.8600  0.0000
  0.0000  0.0000  9.8600
O    H
32   64
Cartesian
  ... (96 atomic positions from Packmol or pre-equilibrated snapshot)
```

**Tip:** Always pre-equilibrate the water structure with a classical force field (e.g., SPC/E in LAMMPS) before starting AIMD. This avoids wasting expensive DFT steps on equilibrating an unphysical initial configuration.

### KPOINTS

For a cell this large (~10 Angstrom), Gamma-point sampling is sufficient.

```
Gamma-point only
0
Gamma
1 1 1
0 0 0
```

### POTCAR

Use the standard PAW potentials:
- `O` (not `O_s` or `O_h`) -- 6 valence electrons
- `H` (not `H_h`) -- 1 valence electron

Concatenate in the same order as species in POSCAR:
```bash
cat $VASP_PP_PATH/O/POTCAR $VASP_PP_PATH/H/POTCAR > POTCAR
```

---
## Trajectory Analysis

The analysis below works on pre-computed trajectory data. After your VASP run completes, copy `XDATCAR` and `OSZICAR` to `data/bulk_water/`.

In [ ]:
import numpy as np
from scipy.stats import linregress
import matplotlib.pyplot as plt
from ase.io import read

plt.rcParams.update({
    'figure.figsize': (6, 4),
    'font.size': 12,
    'axes.linewidth': 1.2,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.width': 1.2,
    'ytick.major.width': 1.2,
})

In [ ]:
# Load the AIMD trajectory
# ASE reads VASP XDATCAR directly; index=':' loads all frames
trajectory = read('data/bulk_water/XDATCAR', index=':')

n_frames = len(trajectory)
n_atoms = len(trajectory[0])
timestep = 1.0  # fs, must match POTIM in INCAR
print(f'Loaded {n_frames} frames, {n_atoms} atoms')
print(f'Total simulation time: {n_frames * timestep / 1000:.1f} ps')

In [ ]:
import nglview as nv
from IPython.display import display

# Interactive 3D view of the initial configuration
view = nv.show_ase(trajectory[0])
view.add_unitcell()
display(view)

# Trajectory playback -- use the slider to scrub through frames
# Subsampled every 10th frame for performance
traj_view = nv.show_asetraj(trajectory[::10])
traj_view.add_unitcell()
traj_view

### Equilibration Check

Before computing properties, verify that the system has equilibrated. The instantaneous temperature should fluctuate around the target value without systematic drift. We parse temperature from the OSZICAR file.

In [ ]:
from pymatgen.io.vasp.outputs import Oszicar

try:
    oszicar = Oszicar('data/bulk_water/OSZICAR')
    # Extract temperature and total energy from MD steps
    temperatures = np.array([step['T'] for step in oszicar.ionic_steps])
    energies = np.array([step['E0'] for step in oszicar.ionic_steps])
except Exception as e:
    print(f"Could not parse OSZICAR: {e}")
    # Fallback to empty arrays if file is missing
    temperatures = np.array([])
    energies = np.array([])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 5), sharex=True)

time_ps = np.arange(len(temperatures)) * timestep / 1000

ax1.plot(time_ps, temperatures, lw=0.5, color='C0')
ax1.axhline(400, ls='--', color='gray', label='Target (400 K)')
ax1.set_ylabel('Temperature (K)')
ax1.legend()

ax2.plot(time_ps, energies, lw=0.5, color='C1')
ax2.set_ylabel('Total energy (eV)')
ax2.set_xlabel('Time (ps)')

plt.tight_layout()
plt.show()

# Discard the first N steps as equilibration
equil_steps = 500
print(f'Mean T (after equil): {temperatures[equil_steps:].mean():.1f} +/- '
      f'{temperatures[equil_steps:].std():.1f} K')

### Radial Distribution Function

The radial distribution function $g(r)$ measures the probability of finding an atom at distance $r$ from another atom, relative to an ideal gas at the same density:

$$g_{AB}(r) = \frac{V}{N_A N_B} \left\langle \sum_{i \in A} \sum_{\substack{j \in B \\ j \neq i}} \delta(r - r_{ij}) \right\rangle \bigg/ 4\pi r^2$$

For liquid water, the key pairs are:
- **O-O**: First peak at ~2.8 Angstrom reveals the hydrogen-bond network
- **O-H**: First peak at ~1.0 Angstrom (intramolecular) and ~1.8 Angstrom (H-bond)
- **H-H**: First peak at ~1.5 Angstrom (intramolecular)

In [ ]:
def compute_rdf(trajectory, pair, r_max=6.0, n_bins=150, start_frame=0):
    """Compute radial distribution function g(r) for a given atom pair.

    Parameters
    ----------
    trajectory : list of ase.Atoms
        MD trajectory frames.
    pair : tuple of str
        Atom types, e.g. ('O', 'O').
    r_max : float
        Cutoff distance (Angstrom). Must be < half the shortest cell vector.
    n_bins : int
        Number of histogram bins.
    start_frame : int
        Skip this many frames for equilibration.

    Returns
    -------
    r : ndarray
        Bin centers (Angstrom).
    g_r : ndarray
        Radial distribution function.
    """
    r_edges = np.linspace(0, r_max, n_bins + 1)
    r_centers = 0.5 * (r_edges[:-1] + r_edges[1:])
    hist = np.zeros(n_bins)

    frames = trajectory[start_frame:]
    n_frames = len(frames)

    # Identify atom indices (same across frames for fixed composition)
    symbols = np.array(frames[0].get_chemical_symbols())
    idx_a = np.where(symbols == pair[0])[0]
    idx_b = np.where(symbols == pair[1])[0]
    same_species = (pair[0] == pair[1])

    for atoms in frames:
        for i in idx_a:
            dists = atoms.get_distances(i, idx_b, mic=True)
            if same_species:
                dists = dists[idx_b != i]
            h, _ = np.histogram(dists, bins=r_edges)
            hist += h

    # Normalize by ideal gas density in each shell
    volume = frames[0].get_volume()
    rho_b = len(idx_b) / volume
    shell_volumes = (4.0 / 3.0) * np.pi * (r_edges[1:]**3 - r_edges[:-1]**3)
    g_r = hist / (n_frames * len(idx_a) * rho_b * shell_volumes)

    return r_centers, g_r

In [ ]:
# Compute RDFs, skipping equilibration frames
r_oo, g_oo = compute_rdf(trajectory, ('O', 'O'), start_frame=equil_steps)
r_oh, g_oh = compute_rdf(trajectory, ('O', 'H'), start_frame=equil_steps)
r_hh, g_hh = compute_rdf(trajectory, ('H', 'H'), start_frame=equil_steps)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

for ax, (r, g, label, color) in zip(axes, [
    (r_oo, g_oo, 'O-O', 'C0'),
    (r_oh, g_oh, 'O-H', 'C1'),
    (r_hh, g_hh, 'H-H', 'C2'),
]):
    ax.plot(r, g, lw=1.5, color=color)
    ax.axhline(1.0, ls='--', color='gray', lw=0.8)
    ax.set_xlabel(r'$r$ ($\mathrm{\AA}$)')
    ax.set_ylabel(r'$g(r)$')
    ax.set_title(label)
    ax.set_xlim(0, 6)
    ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

# Report first peak positions
for name, r, g in [('O-O', r_oo, g_oo), ('O-H', r_oh, g_oh), ('H-H', r_hh, g_hh)]:
    # Find first peak beyond 1.5 A (skip intramolecular for O-H and H-H)
    mask = r > 1.5 if name != 'O-O' else r > 2.0
    peak_idx = np.argmax(g[mask])
    peak_r = r[mask][peak_idx]
    print(f'{name} first intermolecular peak: {peak_r:.2f} A, g = {g[mask][peak_idx]:.2f}')

**Interpreting the RDFs:**
- The O-O $g(r)$ first peak at ~2.8 Angstrom corresponds to the nearest-neighbor hydrogen-bond distance. A pronounced first minimum (~3.3 Angstrom) followed by a second peak (~4.5 Angstrom) indicates tetrahedral ordering.
- The O-H $g(r)$ shows a sharp intramolecular peak at ~1.0 Angstrom and a hydrogen-bond peak at ~1.8 Angstrom.
- Compare your peaks with experimental neutron diffraction data: O-O at 2.77 Angstrom, O-H at 1.85 Angstrom (Soper, 2000).

### Mean Squared Displacement and Diffusion

The mean squared displacement (MSD) tracks how far atoms move over time:

$$\text{MSD}(t) = \left\langle |\mathbf{r}_i(t) - \mathbf{r}_i(0)|^2 \right\rangle$$

In the diffusive regime (long times), MSD grows linearly, and the self-diffusion coefficient follows from the Einstein relation:

$$D = \lim_{t \to \infty} \frac{\text{MSD}(t)}{6t}$$

The factor of 6 applies in 3D. We track oxygen atoms, since they represent the molecular center of mass.

In [ ]:
def compute_msd(trajectory, element='O', start_frame=0):
    """Compute mean squared displacement by unwrapping periodic trajectories.

    Unwraps atomic positions by accumulating frame-to-frame displacements
    with the minimum image convention, avoiding PBC artifacts.
    """
    frames = trajectory[start_frame:]
    symbols = np.array(frames[0].get_chemical_symbols())
    idx = np.where(symbols == element)[0]
    n_frames = len(frames)

    # Build unwrapped positions frame by frame
    unwrapped = np.zeros((n_frames, len(idx), 3))
    unwrapped[0] = frames[0].get_positions()[idx]

    for t in range(1, n_frames):
        prev_pos = frames[t - 1].get_positions()[idx]
        curr_pos = frames[t].get_positions()[idx]
        cell = np.array(frames[t].get_cell())

        # Minimum image displacement
        disp = curr_pos - prev_pos
        frac_disp = disp @ np.linalg.inv(cell)
        frac_disp -= np.round(frac_disp)
        disp = frac_disp @ cell

        unwrapped[t] = unwrapped[t - 1] + disp

    # MSD averaged over all tracked atoms
    r0 = unwrapped[0]
    msd = np.mean(np.sum((unwrapped - r0)**2, axis=2), axis=1)

    return msd

In [ ]:
msd = compute_msd(trajectory, element='O', start_frame=equil_steps)
time_msd = np.arange(len(msd)) * timestep  # in fs

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(time_msd / 1000, msd, lw=1.5, color='C0')
ax.set_xlabel('Time (ps)')
ax.set_ylabel(r'MSD ($\mathrm{\AA}^2$)')
ax.set_title('Oxygen MSD')
plt.tight_layout()
plt.show()

### Extracting the Diffusion Coefficient

We fit the linear portion of the MSD (after the initial ballistic regime, typically $t > 0.5$ ps) to extract $D$.

In [ ]:
# Select the diffusive regime for fitting
t_start_fit = 500   # fs -- skip the ballistic regime
t_end_fit = None     # use all remaining data

mask = time_msd >= t_start_fit
if t_end_fit is not None:
    mask &= time_msd <= t_end_fit

slope, intercept, r_value, _, std_err = linregress(time_msd[mask], msd[mask])

# D = slope / 6 in 3D, units: Angstrom^2 / fs
D_A2_fs = slope / 6.0

# Convert to cm^2/s: 1 A^2/fs = 1e-16 cm^2 / 1e-15 s = 1e-1 cm^2/s
D_cm2_s = D_A2_fs * 1e-1

print(f'Linear fit: slope = {slope:.4f} A^2/fs, R^2 = {r_value**2:.4f}')
print(f'Diffusion coefficient D = {D_cm2_s:.2e} cm^2/s')
print(f'Experimental D(water, 300 K) ~ 2.3e-5 cm^2/s')

# Plot fit
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(time_msd / 1000, msd, lw=1.5, color='C0', label='MSD')
ax.plot(time_msd[mask] / 1000, slope * time_msd[mask] + intercept,
        ls='--', color='C3', lw=1.5, label=f'Fit ($D$ = {D_cm2_s:.2e} cm$^2$/s)')
ax.axvline(t_start_fit / 1000, ls=':', color='gray', lw=0.8, label='Fit start')
ax.set_xlabel('Time (ps)')
ax.set_ylabel(r'MSD ($\mathrm{\AA}^2$)')
ax.legend()
plt.tight_layout()
plt.show()

### Vibrational Density of States (vDOS)

The vibrational density of states $g(\omega)$ can be computed from the Fourier transform of the velocity autocorrelation function (VACF):

$$g(\omega) = \int_0^\infty \frac{\langle \mathbf{v}(t) \cdot \mathbf{v}(0) \rangle}{\langle \mathbf{v}(0) \cdot \mathbf{v}(0) \rangle} e^{i\omega t} dt$$

The peaks in $g(\omega)$ correspond to the characteristic vibrational frequencies of the system (e.g., O-H stretching and bending modes).

In [ ]:
def compute_vdos(trajectory, element='O', start_frame=0, timestep=1.0):
    """Compute the vibrational density of states (vDOS) via VACF.

    Parameters
    ----------
    trajectory : list of ase.Atoms
        MD trajectory frames.
    element : str
        Element to analyze.
    start_frame : int
        Skip equilibration.
    timestep : float
        MD timestep in fs.

    Returns
    -------
    freq : ndarray
        Frequencies (cm^-1).
    vdos : ndarray
        Vibrational density of states (arbitrary units).
    """
    from scipy.fft import fft, fftfreq
    
    frames = trajectory[start_frame:]
    symbols = np.array(frames[0].get_chemical_symbols())
    idx = np.where(symbols == element)[0]
    n_frames = len(frames)
    
    # Extract velocities (if available in trajectory) or estimate from positions
    # For simplicity, we assume the trajectory has velocities stored
    velocities = []
    for atoms in frames:
        v = atoms.get_velocities()
        if v is None:
            raise ValueError("Trajectory does not contain velocities.")
        velocities.append(v[idx])
    velocities = np.array(velocities)
    
    # Compute VACF: <v(t).v(0)> averaged over atoms and time origins
    # Using a simple single-origin VACF for demonstration
    v0 = velocities[0]
    vacf = np.mean(np.sum(velocities * v0, axis=2), axis=1)
    
    # Normalize VACF
    vacf /= vacf[0]
    
    # Fourier Transform
    n_fft = len(vacf)
    vdos = np.abs(fft(vacf))[:n_fft // 2]
    freq = fftfreq(n_fft, d=timestep * 1e-15)[:n_fft // 2]
    
    # Convert frequency from Hz to wavenumbers (cm^-1)
    # freq_cm = freq_Hz / (c * 100)
    c = 299792458
    freq_cm = freq / (c * 100)
    
    return freq_cm, vdos

# Note: vDOS calculation requires velocities in the trajectory.
# Ensure your VASP output includes velocities (standard in XDATCAR or OUTCAR).
try:
    freq_o, vdos_o = compute_vdos(trajectory, element='O', start_frame=equil_steps)
    
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(freq_o, vdos_o, lw=1.5, color='C0')
    ax.set_xlabel(r'Frequency (cm$^{-1}$)')
    ax.set_ylabel('vDOS (arb. units)')
    ax.set_title('Oxygen Vibrational DOS')
    ax.set_xlim(0, 4000)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Could not compute vDOS: {e}")
    print("To enable this, ensure velocities are parsed from the trajectory.")

## Summary


In this tutorial you learned to:
1. Set up VASP input files for NVT AIMD of bulk liquid water
2. Validate equilibration by inspecting temperature and energy time series
3. Compute O-O, O-H, and H-H radial distribution functions and compare with experiment
4. Extract the self-diffusion coefficient from the mean squared displacement

**Next steps:**
- Try different functionals (RPBE, SCAN, r$^2$SCAN) and compare RDF peak positions
- Test convergence with respect to system size (64 H$_2$O) and simulation length
- Add van der Waals corrections (`IVDW = 11` for DFT-D3) and observe the effect on water structure
- Compute the vibrational density of states from the velocity autocorrelation function